# Handling Numerical Data

Numerical data is widely used in data analysis, machine learning, business intelligence, scientific research, and many other fields. From sales figures and customer ages to temperatures, prices, measurements, and performance scores, numerical values help us identify patterns, compare observations, and make data-driven decisions. However, numerical data is rarely ready to use straight away. Real-world datasets often contain missing values, unusual observations, inconsistent scales, or unnecessary rows and columns that need to be addressed before analysis.

In this episode, we take a hands-on approach to handling numerical data. We will work through practical steps including getting basic statistics to understand a dataset, identifying and handling missing values, adding and deleting rows and columns, scaling and normalizing numerical features, and detecting and handling outliers. Rather than focusing heavily on theory, we will apply each technique to a dataset and see how these transformations affect the data, preparing it for further analysis or machine learning.

<div class='alert alert-info'>

:::{objectives}
- Explore and summarize numerical features in the Palmer Penguins dataset.
- Identify and handle missing values in numerical variables.
- Identify and handle outliers in numerical data.
- Apply normalization and standardization techniques to numerical features.
- Prepare clean and appropriately scaled numerical data for machine learning.
:::
</div>

<div class='alert alert-success'>

:::{instructor-note}
- XX minutes teaching
- XX minutes exercising/discussion
:::
</div>

## 1. The Palmer Penguins Dataset

[The Palmer Penguins dataset](https://zenodo.org/records/3960218) is a widely used open dataset in data science and ML education. This dataset contains information on three penguin species that inhabit islands near the Palmer Archipelago in Antarctica: Adelie, Chinstrap, and Gentoo. Each row in the dataset corresponds to a single penguin and records both physical measurements and categorical attributes. The key numerical features include flipper length (mm), culmen length and depth (bill measurements, in mm), and body mass (g). Alongside these, categorical variables such as species, island, and sex are provided.

These data were collected from 2007 - 2009 by Dr. Kristen Gorman with the [Palmer Station Long Term Ecological Research Program](https://lternet.edu/site/palmer-antarctica-lter/), part of the [US Long Term Ecological Research Network](https://lternet.edu/). It contains measurements for 344 penguins from three species -- Adelie, Chinstrap, and Gentoo -- collected from three islands (Biscoe, Dream, and Torgersen) in the Palmer Archipelago, Antarctica.

The data were imported directly from the [Environmental Data Initiative (EDI)](https://edirepository.org/) Data Portal, and are available for use by CC0 license (“No Rights Reserved”) in accordance with the [Palmer Station Data Policy](https://lternet.edu/data-access-policy/).

In [ ]:
from IPython.display import Image, display
display(Image("../images/numerical-data/penguins.png", width=1240))

### 1.1 Loading dataset

Seaborn provides the Penguins dataset through its built-in data-loading functions. We can access it using `sns.load_dataset('penguin')` and then have a quick look at the data.

<div class='alert alert-info'>

:::{note}
If you have your own dataset stored in a CSV file, you can easily load it into Python using Pandas with the `read_csv()` function. This is one of the most common ways to bring tabular data into a DataFrame for further analysis and processing.

Beyond CSV files, Pandas also supports a wide variety of other file formats, making it a powerful and flexible tool for data handling. For example, you can use `read_excel()` to import data from Microsoft Excel spreadsheets, `read_hdf()` to work with HDF5 binary stores, and `read_json()` to load data from JSON files. Each of these formats also has a corresponding method for saving data back to disk, such as `to_csv()`, `to_excel()`, `to_hdf()`, and `to_json()`.
:::
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# penguins = pd.read_csv("./penguins_dataset.csv")

# URL of the Penguins dataset (CSV file)
# url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
# penguins = pd.read_csv(url)

penguins = sns.load_dataset('penguins')
penguins

### 1.2 Getting statistics

Once we have imported the Penguins dataset, our first step is to understand which columns contain numerical data. In this dataset, there are seven columns. Variables such as `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, and `body_mass_g` are numerical, while columns such as `species`, `island`, and `sex` are categorical.

In [ ]:
penguins.info()

For columns with numverical variables, we can summarize using statistics such as the `mean`, `median`, `minimum`, `maximum`, and standard deviation via `.describe()`.

In [ ]:
penguins.describe()

For example, if we look at `body_mass_g`, we might see that the mean is somewhere around 4,200 grams. This tells us the average body mass of the penguins in the available observations. The minimum and maximum tell us the range of observed body masses, while the quartiles tell us how the observations are distributed within that range.

Quartiles divide the ordered data into sections. The 25th percentile, or Q1, tells us the value below which approximately 25% of observations fall. The median divides the data in half, and Q3 marks approximately the 75th percentile. The distance between Q1 and Q3 is called the interquartile range (**IQR**). The IQR is useful because it describes the spread of the middle 50% of our observations and is less influenced by extreme values.

If the `mean` and `median` (50%) are relatively close, that can suggest that the distribution is reasonably balanced. If they are noticeably different, it may indicate that the data is skewed or that some unusually large or small observations are affecting the `mean`.

### 1.3 Data visualization

Looking only at the raw numbers in the `penguins` DataFrame, or even examining the statistical summaries provided by `penguins.info()` and `penguins.describe()`, often does not give us a clear intuition about the patterns and relationships in the data. To truly understand the dataset, we generally prefer to visualize the data, since graphical representations can reveal trends, groupings, and anomalies that may remain hidden in numerical summaries alone.

One nice visualization for datasets with relatively few attributes is **Pair Plot**, which can be created using `sns.pairplot(...)`. It shows a scatterplot of each attribute plotted against each of the other attributes. By using the `hue='species'` setting for the pairplot the graphs on the diagonal are layered kernel density estimate plots for the different values of the `species` column.

In [ ]:
sns.pairplot(penguins, hue="sex", height=2.0)

In [ ]:
sns.pairplot(penguins, hue="island", height=2.0)

In [ ]:
sns.pairplot(penguins, hue="species", height=2.0)

<div class='alert alert-warning'>

:::{discussion}
Take a look at the pairplot we created. Consider the following questions:
- Is there any class that is easily distinguishable from the others?
- Which combination of attributes shows the best separation for all 3 class labels at once?
- For pairplot with ``hue="sex"``, which combination of features distinguishes the two sexes best?
- What about the one with ``hue="island"``? 
</div>

## 2. Imputating Missing Values

After getting a statistical overview of our numerical variables, the next issue we need to investigate is missing data. In a real-world dataset, it is common for some measurements to be unavailable. In the Penguins dataset, for example, some penguins may have a missing `body_mass_g`, `flipper_length_mm`, or another measurement.

Missing values matter because many calculations and machine-learning algorithms cannot work directly with incomplete data. Before deciding what to do with missing values, however, we first need to find them and understand how much data is actually missing. The important point is that we should inspect the missing data before automatically deleting or replacing anything.

In [ ]:
# check whether each column contains missing values
penguins.isna().sum()

In [ ]:
# total missing values in the entire dataset
penguins.isna().sum().sum()

To focus specifically on numerical columns:

In [ ]:
numericals = ["body_mass_g", "bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
penguins_numericals = penguins[numericals].copy()
penguins_numericals.isna().sum()

In [ ]:
penguins_numericals

### 2.1 Dropping missing observations

Sometimes we only care about missing values in a particular numerical variable. For example, suppose we want to  
This is useful because it allows us to inspect the actual observations rather than treating missing values as abstract numbers.

At this point, we should ask an important question: Should we delete these rows, or should we fill in the missing values?
One straightforward approach is to remove rows containing missing values. Pandas provides `.dropna()` for this.

In [ ]:
print("Original shape:", penguins_numericals.shape)
penguins_numericals_dropna = penguins_numericals.dropna()
print("New shape:", penguins_numericals_dropna.shape)

penguins_numericals_dropna.describe()

<div class='alert alert-warning'>

:::{warning}
The `dropna()` is easy to use, but "easy" does not necessarily mean "appropriate".
The decision to drop rows should therefore depend on the situation. If only a small number of observations are missing and those observations are not critical, dropping them may be perfectly reasonable. But if many observations are missing, deleting them could significantly reduce our dataset and potentially introduce bias.
:::
</div>

### 2.2 Imputation of missing values

Another option is to impute the missing values with a suitable estimate. Common choices include the `mean` or `median` of that column, depending on the distribution.
Here we calculate `mean` and `median` values for numerical features. To illustrate this process, we take `body_mass_g` feature as an example.

In [ ]:
body_mass_g_mean = penguins_numericals.body_mass_g.mean()
body_mass_g_median = penguins_numericals.body_mass_g.median()
print(f"  mean value of body_mass_g is {body_mass_g_mean}")
print(f"median value of body_mass_g is {body_mass_g_median}")

Rather than directly replacing missing values, we concatenate new columns containing `mean` and `median` values for the `body_mass_g` feature to end of Penguins dataset, and than visualize distribution of this feature.

In [ ]:
penguins_numericals['BMG_mean'] = penguins_numericals.body_mass_g.fillna(body_mass_g_mean)
penguins_numericals['BMG_median'] = penguins_numericals.body_mass_g.fillna(body_mass_g_median)
penguins_numericals.head().style.highlight_null(color = 'green')

In [ ]:
plt.figure(figsize=(9, 6))

penguins_numericals['body_mass_g'].plot(kind='kde', color='tab:green', label="body_mass_g")
penguins_numericals['BMG_mean'].plot(kind='kde', color='tab:orange', label="BMG_mean")
penguins_numericals['BMG_median'].plot(kind='kde', color='tab:blue', label="BMG_median")

plt.legend(loc='best')
plt.tight_layout()

<div class='alert alert-warning'>

:::{callout} Mean or median imputation
- Mean imputation is simple and can work well when the data is reasonably balanced and the missing values are not strongly associated with unusual observations.
    - But there is an important limitation: the mean can be affected by extreme values. If a numerical variable is strongly skewed or contains outliers, the mean may not represent a typical observation very well.
- In such situation, the median becomes particularly useful. The median is less sensitive to extreme values than the mean, so it can often be a better choice when numerical data is skewed or contains outliers.

The key lesson is that missing-value handling is not simply a technical step. We need to make a decision based on how much data is missing, where it is missing, and what effect removing or replacing it might have on our analysis.
:::
</div>

## 3. Handling Outliers

After getting statistical information from the Penguins dataset, the next step is to look more carefully at the unusual observations.

An **outlier** is a value that is unusually far from the other observations. For instance, if body mass of most of penguins in dataset varies between 3000-6000 g, an observation of 7500 g will be considered as an outlier since such an observation occurs rarely.

Outliers can occur because of measurement errors, data-entry mistakes, unusual but genuine cases, or because the population itself naturally contains extreme observations. Therefore, we should not automatically delete an outlier just because it looks unusual. Instead, we first identify it, investigate it, understand its impact, and then decide whether to retain, remove, or transform it.

In [ ]:
penguins_numericals

### 3.1 Imputating missing values with extreme data

In the code snippet below, we create a new column for `body_mass_g` and impute its missing values using the extreme **end of the distribution** (EoD). Here, the EoD is defined as the mean plus three standard deviations, calculated as `mean + (3 * std)`.

In [ ]:
eod_value = penguins['body_mass_g'].mean() + 3 * penguins['body_mass_g'].std() + 200
print(eod_value)

penguins_numericals['species'] = penguins['species']
penguins_numericals['BMG_EoD'] = penguins_numericals['body_mass_g'].fillna(eod_value)
penguins_numericals.head(5).style.highlight_null(color = 'green')

In [ ]:
plt.figure(figsize=(8, 5))

sns.swarmplot(y=penguins_numericals["species"], x=penguins_numericals["BMG_EoD"],
              color="lightgreen", marker="o", size=4)

sns.boxplot(y=penguins_numericals["species"], x=penguins_numericals["BMG_EoD"], 
            palette="coolwarm", notch=True, linewidth=2, width=0.5, 
            hue=penguins_numericals.species, legend=False)

plt.xlabel("Body mass (g)", fontsize=14)
plt.ylabel("Species", fontsize=14)
plt.tick_params(axis='x', labelsize=12)
plt.tick_params(axis='y', labelsize=12)

plt.tight_layout()

There are several approaches to identify outliers, and two of most commonly used methods are **Interquartile Range (IQR)** method and **Mean-Standard Deviation** method.

### 3.2 The Inter quartile range (IQR) method

IQR measures spread of middle 50% of data and is calculated by subtracting 1st quartile (25th percentile, Q1) from 3rd quartile (75th percentile, Q3).
- Once IQR is obtained, we can determine boundaries for detecting outliers.
- Lower limit is defined as Q1 minus 1.5 times IQR.
- Upper limit is defined as Q3 plus 1.5 times IQR.

In [ ]:
penguins_numericals.body_mass_g.describe()

In [ ]:
print(f"25% quantile = {penguins_numericals['body_mass_g'].quantile(0.25)}")
print(f"75% quantile = {penguins_numericals['body_mass_g'].quantile(0.75)}\n")

IQR = penguins_numericals["body_mass_g"].quantile(0.75) - penguins_numericals["body_mass_g"].quantile(0.25)
lower_bmg_limit = penguins_numericals["body_mass_g"].quantile(0.25) - (1.5 * IQR)
upper_bmg_limit = penguins_numericals["body_mass_g"].quantile(0.75) + (1.5 * IQR)

print(f"lower limt of IQR = {lower_bmg_limit} and upper limit of IQR = {upper_bmg_limit}")

Any data points that fall below lower limit or above upper limit are considered outliers, and these points are subsequently removed from dataset.

In [ ]:
# check data points larger than the upper limit of the IQR range
penguins_numericals[penguins_numericals["BMG_EoD"] > upper_bmg_limit]

In [ ]:
# check data points smaller than the lower limit of the IQR range
penguins_numericals[penguins_numericals["body_mass_g"] < lower_bmg_limit]

<div class='alert alert-danger'>

:::{questions} **What should we actually do with the outliers?**
- It is tempting to say, "We found outliers, so let's delete them." But that isn't necessarily the right decision.
- Imagine that an unusually heavy penguin is a genuine measurement.
    - Removing it would mean throwing away valid information.
    - On the other hand, if we discover that a measurement was entered incorrectly. For example, 35,000 grams instead of 3,500 grams, then removing or correcting that observation may be appropriate.
- The decision can be thought of as three possibilities:
    - Retain: The value is unusual but valid.
    - Remove: The value is incorrect, impossible, or inappropriate for the analysis.
    - Transform or cap: The value is valid, but its extreme magnitude has too much influence on the analysis.
- > This is why an outlier is not automatically an error.
:::
</div>

<div class='alert alert-warning'>

:::{callout} Capping or winsorizing
Instead of deleting extreme observations, another option is to cap them. This means we keep the observation but limit its value to a predefined boundary.
For the two abnormal values in `BMG_EoD` column, values below the lower boundary are replaced by the lower boundary, and values above the upper boundary are replaced by the upper boundary.
This approach is sometimes called **winsorizing** when extreme values are replaced with specified percentile-based limits.
    - The advantage is that we don't lose entire observations.
    - The disadvantage is that we are changing the numerical values, so we need to make sure that this transformation makes sense for our particular analysis.
:::
</div>

## 4. Scaling & Normalizing Numerical Data

In many datasets, numerical features can be measured in very different units and ranges.
For example, in the Penguins dataset, body mass is measured in grams and can range in the thousands, while flipper length is measured in millimeters and typically ranges in the hundreds.
Both are numerical variables, but their numerical scales are very different. This difference can matter when we use the data for certain types of analysis and machine learning, making features with larger values dominate the learning process, leading to biased and inaccurate models.

Scaling transforms these features to a common, limited range, such as [0, 1] or a distribution with a mean of 0 and a standard deviation of 1, without distorting the differences in the ranges of values or losing information.

In this section, we will see why scale matters and then apply two common transformations: **standardization** (Z-score normalization) and **normalization** (min-max scaling). We will compare the original and transformed values so that we can see exactly what these techniques do.

### 4.1 Standardization (Z-score Normalization)

One of the most common approaches is standardization. Standardization transforms a variable so that it has a mean close to 0 and a standard deviation close to 1.
It is calculated by subtracting the mean value ($\mu$) of the feature and then dividing by the standard deviation ($\sigma$), and its formula is $X\_scaled = \frac{X - \mu}{\sigma}$.

In [ ]:
penguins_BMG = penguins[["body_mass_g"]].copy()
penguins_BMG.dropna(inplace=True)
print(penguins_BMG)

We use the `StandardScaler` from scikit-learn to standardize the `body_mass_g` variable. It transforms the data by subtracting the mean and dividing by the standard deviation, resulting in a mean of approximately 0 and a standard deviation of 1. This ensures that the variable is on a consistent scale for machine learning algorithms.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
penguins_BMG["BMG_Std"] = scaler.fit_transform(penguins_BMG[["body_mass_g"]])
penguins_BMG

In [ ]:
# verify the mean and standard deviation of standardized BMG data
print("mean value:", penguins_BMG["BMG_Std"].mean())
print("standard deviation:", penguins_BMG["BMG_Std"].std())

### 4.2 Normalization (Min-Max Scaling)

Standardization is not the only option. Another common approach is **min-max normalization**, which transforms values into a specified range, typically [0, 1].
It is calculated by subtracting the minimum value of the feature and then dividing by the range (max - min), and its formula is $X\_scaled = \frac{(X - X\_min)}{(X\_max - X\_min)}$.

This method is useful when the distribution is not Gaussian or when the algorithm requires input values bounded within a specific range (*e.g.*, neural networks often use activation functions that expect inputs in the [0,1] range).

In [ ]:
from sklearn.preprocessing import MinMaxScaler
minmax_scaler = MinMaxScaler()

penguins_BMG["BMG_MinMax"] = minmax_scaler.fit_transform(penguins_BMG[["body_mass_g"]])
penguins_BMG

In [ ]:
# verify the min and max values of normalized BMG data
print("min value:", penguins_BMG["BMG_MinMax"].min())
print("max value:", penguins_BMG["BMG_MinMax"].max())

<div class='alert alert-warning'>

:::{callout} Normalization vs. Standardization

- Normalization, such as Min-Max scaling, is useful when features need to be mapped to a specific range, typically between 0 and 1, but it can be **sensitive to outliers** because extreme values affect the minimum and maximum. 
- Standardization transforms data to have a mean of 0 and a standard deviation of 1, making it generally more robust to differences in feature scales, and is less affected by outliers, although it does not restrict values to a fixed range.

Both methods can improve model performance and convergence, but the appropriate choice depends on the data distribution and the requirements of the machine learning algorithm.
:::
</div>

<div class='alert alert-success'>

:::{exercise}
After completing the individual tasks for handling numerical data, it is now time to bring everything together in a hands-on exercise. In this activity, we will work with representative datasets from scikit-learn and apply the data-handling techniques we have learned throughout the episode.

- **Select a dataset**: Choose one dataset from scikit-learn: Iris, Diabetes, Wine, Breast Cancer, or California Housing.
    ```python
    from sklearn import datasets

    iris = datasets.load_iris(as_frame=True)
    iris_df = iris.frame
    iris_df.head()

    diabetes = datasets.load_diabetes(as_frame=True)
    diabetes_df = diabetes.frame
    diabetes_df.head()

    wine = datasets.load_wine(as_frame=True)
    wine_df = wine.frame
    wine_df.head()

    cancer = datasets.load_breast_cancer(as_frame=True)
    cancer_df = cancer.frame
    cancer_df.head()

    housing = datasets.fetch_california_housing(as_frame=True)
    housing_df = housing.frame
    housing_df.head()
    ```
- **Inspect the dataset**: Examine the shape, columns, data types, and first few observations.
- **Remove unnecessary rows and columns**: Identify irrelevant or redundant data and remove it where appropriate.
- **Get statistical information**: Calculate and interpret count, mean, median, minimum, maximum, quartiles, and standard deviation.
- **Check for missing values**: Identify missing values and calculate their counts and percentages.
- **Decide how to handle missing values**: Determine whether to remove observations or impute missing numerical values using an appropriate method.
- **Identify and handle outliers**: Use summary statistics, the IQR method, and visualizations to identify potential outliers, then decide whether to retain, remove, or cap them.
- **Scale or normalize numerical features**: Apply standardization or min-max normalization to selected features and compare the values before and after transformation.
- **Verify the cleaned dataset**: Check the final shape, columns, data types, missing values, ranges, and other relevant properties to confirm that the dataset is ready for further analysis or machine learning.
:::
</div>

<div class='alert alert-info'>

:::{keypoints}
- Get familiar with the Palmer Penguins dataset.
- Perform statistical analysis and data visualization using Pandas and Seaborn.
- Identify and handle missing values in numerical features.
- Identify and handle outliers in the dataset.
- Apply standardization and normalization techniques to numerical features.
:::